In [ ]:
import strahler
from importlib.metadata import version as package_version

package = package_version("strahler")
library = strahler.version()
print(f"strahler package {package}; libstrahler {library}")
if package != library:
    raise RuntimeError("Package/library version mismatch. Reinstall strahler in this kernel, "
                       "check STRAHLER_LIB, and restart the kernel before continuing.")

from notebook_support import HERE, BACKENDS, metrics_table, table
from strahler.widgets import GameView, TreeView, Stepper
from IPython.display import display

# Check your own decomposition

Write a tree using `{h,H,T,S,children}` and external vertex IDs. Bounds
use normalized solver priorities. The independent checker returns a condition,
node path, and vertex; it does not repair the tree or assume the solver agrees.
The example has two nested priority levels. It breaks and repairs the child's
bound, then locates a top-priority vertex failure.

In [ ]:
from copy import deepcopy
game = strahler.Game.from_text('0 2 0 0 "high"; 1 0 0 1 "low";')
good = dict(h=2, H=[0], T=[], S=[], children=[
    dict(h=0, H=[1], T=[], S=[], children=[])])
bad = deepcopy(good)
bad["children"][0]["h"] = 2
failure = strahler.check(game, bad, 0)
assert not failure.ok and failure.path == (0,) and failure.condition is strahler.Condition.CHILD
print("Child bound failure:", failure)
view = GameView(game)
display(view, TreeView(bad, view))
bad["children"][0]["h"] = 0
fixed = strahler.check(game, bad, 0)
assert fixed.ok
print("Repaired tree:", fixed)
display(TreeView(bad, view))
wrong_top = dict(h=0, H=[0, 1], T=[], S=[], children=[])
located = strahler.check(game, wrong_top, 0)
assert not located.ok and located.vertex == 0 and located.condition is strahler.Condition.PRIORITY
view.highlight([located.vertex], "#fecaca", "checker failure")
print("Top-priority vertex failure:", located)

Try a JSON tree in the editor below. "Check tree" reports malformed input or
the independent verdict, including failure locations. The initial check runs
automatically. For a single winning region, pass that region as `subgame=`;
the induced subgame must be total. An empty restriction is valid.

In [ ]:
import json
from ipywidgets import Textarea, Button, Output, VBox
field = Textarea(value=json.dumps(good, indent=2), description="Tree JSON", layout={"width": "100%", "height": "240px"})
button = Button(description="Check tree")
output = Output()
def inspect_tree(_=None):
    with output:
        output.clear_output(wait=True)
        try:
            tree = json.loads(field.value)
            verdict = strahler.check(game, tree, 0)
            print(verdict)
            display(TreeView(tree, view))
        except (ValueError, strahler.StrahlerError) as error:
            print(f"Invalid tree: {error}")
button.on_click(inspect_tree)
inspect_tree()
display(VBox([field, button, output]))